In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numba import njit
from scipy.stats import chi2

import src.data_loader as dl
from src.datasets import LatLonData, XYData, merge, project_latlon_data
from src.preclustering import precluster_latlon_boxes
from src.density_clustering import density_clustering
from src.gmm import gmm
from src.utils import min_enclosing_cap
from src.plotting import generate_folium_map, plot_data

%matplotlib inline

## Load data

In [ ]:
filepath = 'example 2 months rot.csv'
df, nans = dl.load_data(filepath)

In [ ]:
# Create dict of ElNots + counts, sorted in descending order

elnots = df['elnot']
elnots, counts = np.unique(elnots, return_counts=True)
elnots = {elnot: int(count) for elnot, count in zip(elnots, counts)}
elnots = dict(sorted(elnots.items(), key=lambda item: item[1], reverse=True))
print(elnots)

### Load elnot subset as `LatLonData` instance

In [ ]:
elnot = 'B080Q'
# elnot = 'J349D'
data = df[df['elnot'] == elnot]
latlons, ellipses = dl.extract_arrays(data, names=['latlons', 'ellipses'])

data = LatLonData(latlons, ellipses, 
                  idx=data.index.astype(int))


## Precluster 1
### Split per bounding boxes connected components

In [ ]:
zones, _ = precluster_latlon_boxes(data, check_overlap=True)

In [ ]:
# Analyze zones
split_zone_lengths = [len(zone) for zone in zones]
zone_radii = [min_enclosing_cap(zone.latlons)[1] for zone in zones]
zone_radii = ', '.join([f"{r:.2f}" for r in zone_radii])

print(f"\nAfter initial split into {len(zones)} zones by bbox overlap:")
print(f"Zone sizes [# points]: {split_zone_lengths}")
print(f"Zone radii [degrees]: {zone_radii}")

## Generate folium maps html files
Too large datasets take some time to generate and might crash your browser when opening.

In [ ]:
# merged_data = merge(zones)
# # Create dir if not exists: maps
# import os
# if not os.path.exists('maps'):
#     os.makedirs('maps')

# m = generate_folium_map(merged_data, mode='bbox', tiles=None)
# m.save('maps/bbox.html')

# m = generate_folium_map(merged_data, mode='ellipse', tiles=None)
# m.save('maps/ellipses.html')

# m = generate_folium_map(zones[0], mode='points')
# m.save('maps/zone_1.html')

## Projection
Project zone to 2D data (`XYData` instance) and plotting\
Note: Plotting ellipses for large datasets takes some time

In [ ]:
xy_zones = [project_latlon_data(zone) for zone in zones]

In [ ]:
fig, ax = plot_data(xy_zones[0], show_ellipses=True)
plt.show()

In [ ]:
testset = xy_zones[0]
subsets, outliers, peaks = density_clustering(testset, 
                                              min_dist_between_peaks=700,
                                              sigma_in_meters=1000,
                                              threshold_count_per_km2=400,
                                              verbose=True,
                                              outliers_subgroup_tol=0)

labeled_testset = XYData.merge(subsets)

In [ ]:
subset = subsets[0]
len(subset)

In [ ]:
A = subset.compute_adjacency_matrix()
D = np.diag(A.sum(axis=1))
L = D - A
Lnorm = np.linalg.inv(D)**0.5 @ L @ np.linalg.inv(D)**0.5
vals, vecs = np.linalg.eigh(Lnorm)

In [ ]:
eig_gaps = np.diff(vals)
sort_idx = np.argsort(eig_gaps)[::-1]
K = sort_idx[0] + 1
K

In [ ]:
for i, subset in enumerate(subsets):
    fig, ax = plot_data(subset, figsize=(16,12))
    fig.savefig(f'img/cluster_{i}.png')
    plt.close(fig)
    # generate_folium_map(data.get_by_index(subset.idx), 
    #                     ellipse_num_points=10,
    #                     ellipse_alpha=0.1).save(f'maps/cluster_{i}.html')

In [ ]:
to_plot = labeled_testset.extract_bounding_box(x_max=0, y_max=0)
# to_plot = to_plot[np.isin(to_plot.labels, selected_labels)]

plot_data(to_plot, show_ellipses=True, figsize=(16,12), ellipses_alpha=.5)
# plt.scatter(peaks[:,0]/1000, peaks[:,1]/1000, c='red', s=200, marker='x', label='Blob centers')

# plot_data(outliers, show_ellipses=True, figsize=(16,12), ellipses_alpha=.5)


# plt.xlim(-55, -46)
# plt.ylim(-192, -182)
plt.show()

In [ ]:
folium_data = data.get_by_index(to_plot.idx)

In [ ]:
len(labeled_testset.parent)

In [ ]:
# 1. Get the indices that would sort A.idx
sorter = np.argsort(to_plot.idx)

# 2. Find the positions of B.idx elements within the sorted A.idx
#    This gives us an array of indices that maps B.idx to the sorted A.idx
positions = np.searchsorted(to_plot.idx, folium_data.idx, sorter=sorter)

# 3. Use the sorter to get the final indices into the original A.labels
final_indices = sorter[positions]

# 4. Assign the correctly ordered labels from A to B
folium_data.labels = to_plot.labels[final_indices]

In [ ]:
m = generate_folium_map(folium_data, mode='points', ellipse_num_points=12)
m.save('maps/labeled_set_points.html')

## Second split
Compute weighted graph based on pairwise mahalanobis distances and split into connected components.

In [ ]:
import cProfile
import pstats

In [ ]:
pr = cProfile.Profile()
pr.enable()
subset.split_into_connected_components()
pr.disable()

In [ ]:
stats = pstats.Stats(pr)
stats.sort_stats('cumulative')
stats.print_stats(20)

In [ ]:
len(subset)

## Final clustering
1. Spectral clustering for K (# clusters) and first labels assignment. This one reuses the previously computed graph
2. Adapted GMM for fine-tuned output.

In [ ]:
gmm_sets = []
for gmm_set in subset.split_by_labels():
    gmm_set.spectral_clustering()
    gmm(gmm_set)
    gmm_sets.append(gmm_set)

subset = XYData.merge(gmm_sets)

In [ ]:
plot_data(subset, figsize=(16,9), show_ellipses=True)

In [ ]:
subset.labels